# SPU Stormwater EDA — v01

Starter notebook for exploring Seattle Public Utilities' stormwater asset inventory. Two samples are bundled with this notebook, each suited to a different purpose:

| Sample | Location | Purpose |
|---|---|---|
| **`sample/`** | `~/Desktop/spu/sample/` | **Random** sample (~8.7K features, 12 layers). Best for cross-sectional analysis — counts, distributions, ownership splits — where statistical representativeness matters. |
| **`sample_geo/`** | `~/Desktop/spu/sample_geo/` | **Geographic** sample (~18K features, 23 layers) of every asset within a Pioneer Square / SODO bounding box. Best for **mapping** — pipes connect, basins overlap, the network looks like a network. |

Each row in either sample carries the original SPU attributes plus two geometry columns:

| Column | Purpose |
|---|---|
| `geometry_wkt` | Well-Known Text geometry. Universal portable format. |
| `geometry_type` | `Point`, `LineString`, or `Polygon`. |

After running section 5b, every layer also gets `lat` / `lon` columns derived from the geometry.

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
from shapely import from_wkt

import folium
from folium.plugins import MarkerCluster
import random
from shapely import from_wkt
import geopandas as gpd
import webbrowser, os

import folium 

import matplotlib.pyplot as plt

SAMPLE     = Path.home() / "spu" / "sample"      # random sample (cross-sectional)
SAMPLE_GEO = Path.home() / "spu" / "sample_geo"  # geographic sample (visualization)
GEOMETRY_COLS = ("geometry_wkt", "geometry_type")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 1. Sample manifest (random sample)

`sample/_index.csv` is the cheat sheet for the random sample — what's in the folder, asset class, geometry type, and how the sample compares to the full layer.

In [4]:
index = pd.read_csv(SAMPLE / "_index.csv")
index

,file,asset_class,label,geometry_type,n_features_in_sample,n_features_in_full,n_attributes,size_kb
0,DWW_Pump_Stationsss__0.csv,Pump Stations,Pump Stations,Polygon,141,141,36,220
1,SPU_DWW_Appurtenance_Valves__0.csv,Appurtenances,Appurtenance Valves,Point,384,384,47,127
2,SPU_DWW_CSO_Basins___0.csv,Service Areas,CSO Basins,Polygon,81,81,9,565
3,SPU_DWW_Catch_Basins_Junction_Boxes_Sandboxes_...,Collection,Catch Basins (sample),Point,1500,91965,34,459
4,SPU_DWW_Ditches_and_Culverts_DSO_EXT_View__1.csv,Conveyance,Ditches & Culverts (sample),LineString,1000,26741,41,336
5,SPU_DWW_Green_Stormwater_Infrastructure__2.csv,Green Infrastructure,GSI Weirs,Point,169,169,37,48
6,SPU_DWW_Green_Stormwater_Infrastructure__3.csv,Green Infrastructure,GSI Swales (sample),LineString,1000,12743,31,277
7,SPU_DWW_Mainlines_External__10.csv,Conveyance,Mainlines Permitted Use (sample),LineString,1500,65234,56,724
8,SPU_DWW_Ponds__0.csv,Detention,Ponds,Polygon,53,53,24,113
9,SPU_DWW_Storm_Outfalls__0.csv,Outfalls,Storm Outfalls,Point,2472,2472,44,751


## 2. Load all (random) sample layers

Stored as a `dict` keyed by the human-readable label. Grab one with `data["Storm Outfalls"]` or iterate.

In [5]:
data = {row["label"]: pd.read_csv(SAMPLE / row["file"], low_memory=False) for _, row in index.iterrows()}
print(f"Loaded {len(data)} layers, {sum(len(d) for d in data.values()):,} total features")
for label, df in data.items():
    print(f"  {label:<40} {len(df):>6,} rows  x  {len(df.columns):>3} cols")

Loaded 12 layers, 8,696 total features
  Pump Stations                               141 rows  x   38 cols
  Appurtenance Valves                         384 rows  x   49 cols
  CSO Basins                                   81 rows  x   11 cols
  Catch Basins (sample)                     1,500 rows  x   36 cols
  Ditches & Culverts (sample)               1,000 rows  x   43 cols
  GSI Weirs                                   169 rows  x   39 cols
  GSI Swales (sample)                       1,000 rows  x   33 cols
  Mainlines Permitted Use (sample)          1,500 rows  x   58 cols
  Ponds                                        53 rows  x   26 cols
  Storm Outfalls                            2,472 rows  x   46 cols
  Underground Injection Cells                  32 rows  x   21 cols
  Vaults                                      364 rows  x   27 cols


## 3. Confirm geometry fields are present and populated

Sanity check: each layer should have a non-null `geometry_wkt` for every row, with a consistent `geometry_type`.

In [6]:
rows = []
for label, df in data.items():
    rows.append({
        "layer": label,
        "rows": len(df),
        "attribute_cols": len(df.columns) - len(GEOMETRY_COLS),
        "geometry_type": df["geometry_type"].iloc[0] if len(df) else "",
        "wkt_populated": df["geometry_wkt"].notna().sum(),
        "sample_wkt": (df["geometry_wkt"].iloc[0][:55] + "...") if len(df) and pd.notna(df["geometry_wkt"].iloc[0]) else "",
    })
pd.DataFrame(rows)

,layer,rows,attribute_cols,geometry_type,wkt_populated,sample_wkt
0,Pump Stations,141,36,Polygon,141,"POLYGON ((-122.393974485575 47.5231245776586, ..."
1,Appurtenance Valves,384,47,Point,384,POINT (-122.414186629566 47.6450991349859)...
2,CSO Basins,81,9,Polygon,81,"POLYGON ((-122.26553246144 47.5598632676095, -..."
3,Catch Basins (sample),1500,34,Point,1500,POINT (-122.384254639804 47.5609790351001)...
4,Ditches & Culverts (sample),1000,41,LineString,1000,LINESTRING (-122.321439205202 47.7542264181995...
5,GSI Weirs,169,37,Point,169,POINT (-122.37757003302 47.6477274713008)...
6,GSI Swales (sample),1000,31,LineString,1000,LINESTRING (-122.387666971873 47.5649605829734...
7,Mainlines Permitted Use (sample),1500,56,LineString,1500,LINESTRING (-122.295107859047 47.5843878692265...
8,Ponds,53,24,Polygon,53,"POLYGON ((-122.323484158948 47.7029747221754, ..."
9,Storm Outfalls,2472,44,Point,2472,POINT (-122.268347750822 47.6582186906274)...


## 4. Cross-sectional exploration (random sample)

For analysis that doesn't need geometry — counts, distributions, correlations — drop the geometry columns for a cleaner view. Below are starter probes; replace with whatever you want to investigate.

### 4a. Storm Outfalls — ownership breakdown

In [7]:
outfalls = data["Storm Outfalls"]
outfalls_attrs = outfalls.drop(columns=list(GEOMETRY_COLS))
print(f"Storm outfalls — {len(outfalls):,} rows, {len(outfalls_attrs.columns)} attribute columns")
outfalls["OUT_OWNER_NAME"].value_counts(dropna=False).head(10)

Storm outfalls — 2,472 rows, 44 attribute columns


OUT_OWNER_NAME
Private                     797
Seattle Public Utilities    675
Parks and Recreation        348
Unknown                     213
Port of Seattle             206
King County                  62
US Government                41
University of Washington     39
Tukwila SA                   32
State of Washington          26
Name: count, dtype: int64

### 4b. Mainlines — pipe material distribution

In [8]:
mains = data["Mainlines Permitted Use (sample)"]
material_col = next((c for c in mains.columns if "MTRL" in c.upper() and "TEXT" in c.upper()), None)
print(f"Using material column: {material_col}")
if material_col:
    mains[material_col].value_counts(dropna=False).head(10)

Using material column: None


## 5. Mapping setup — parse WKT to real geometries

1. `shapely.from_wkt` converts WKT strings into geometry objects.
2. Wrap in `geopandas.GeoDataFrame` for spatial joins, projection handling, and built-in plotting.
3. Hand to a mapping library — `folium`, `contextily`, `kepler.gl`, etc.

If the libraries aren't installed yet:

```bash
pip install shapely geopandas folium
```

All sample geometries are in **WGS84 (EPSG:4326)** — lon/lat decimal degrees.

In [9]:
try:
    from shapely import from_wkt
    import geopandas as gpd
    HAS_GIS = True

    def to_gdf(df: pd.DataFrame, crs: str = "EPSG:4326") -> "gpd.GeoDataFrame":
        """Parse `geometry_wkt` into shapely geometries and return a GeoDataFrame."""
        d = df.copy()
        d["geometry"] = d["geometry_wkt"].apply(from_wkt)
        return gpd.GeoDataFrame(d, geometry="geometry", crs=crs)

    print("GIS libraries available. `to_gdf(df)` helper is ready.")
except ImportError as e:
    HAS_GIS = False
    print(f"Mapping libraries not installed: {e}")
    print("Run:  pip install shapely geopandas folium")

GIS libraries available. `to_gdf(df)` helper is ready.


## 5b. Derive uniform `lat` / `lon` on every layer

Adds `lat` and `lon` columns to every random-sample layer:

- **Points**: lat / lon are the point coordinates directly.
- **Lines / Polygons**: lat / lon are the geometry's centroid.

In [10]:
if HAS_GIS:
    def add_lat_lon(df: pd.DataFrame) -> pd.DataFrame:
        """Add `lat` / `lon` columns derived from `geometry_wkt`.
        Points -> point coords; lines/polygons -> centroid."""
        d = df.copy()
        geoms = d["geometry_wkt"].apply(lambda w: from_wkt(w) if pd.notna(w) else None)
        centroids = geoms.apply(lambda g: g.centroid if g is not None else None)
        d["lat"] = centroids.apply(lambda c: c.y if c is not None else None)
        d["lon"] = centroids.apply(lambda c: c.x if c is not None else None)
        return d

    data = {label: add_lat_lon(df) for label, df in data.items()}

    print(f"{'layer':<40} {'lat/lon%':>9}  {'lat range':<22} {'lon range':<22}")
    print("-" * 100)
    for label, df in data.items():
        ok = df[['lat', 'lon']].notna().all(axis=1).mean() * 100
        print(f"{label:<40} {ok:>8.1f}%  "
              f"({df['lat'].min():.4f}, {df['lat'].max():.4f})  "
              f"({df['lon'].min():.4f}, {df['lon'].max():.4f})")
else:
    print("Run the mapping-setup cell first (shapely is needed).")

layer                                     lat/lon%  lat range              lon range             
----------------------------------------------------------------------------------------------------
Pump Stations                               100.0%  (47.4965, 47.7628)  (-122.4192, -122.2438)
Appurtenance Valves                         100.0%  (47.5086, 47.7323)  (-122.4169, -122.2537)
CSO Basins                                  100.0%  (47.5072, 47.6813)  (-122.4195, -122.2503)
Catch Basins (sample)                       100.0%  (47.4193, 47.7539)  (-122.4176, -121.7679)
Ditches & Culverts (sample)                 100.0%  (47.3898, 47.8562)  (-122.4268, -121.7642)
GSI Weirs                                   100.0%  (47.5229, 47.7179)  (-122.3897, -122.3137)
GSI Swales (sample)                         100.0%  (47.5078, 47.7337)  (-122.4186, -122.2558)
Mainlines Permitted Use (sample)            100.0%  (47.4884, 47.7654)  (-122.4178, -122.2192)
Ponds                                    

## 5c. Load the geographic sample (for mapping)

Random sampling spreads features thin — useful for distributions but produces a sparse, disconnected map. The geographic sample at `~/Desktop/spu/sample_geo/` instead keeps **every** asset within a Pioneer Square / SODO / Elliott Bay bounding box. Same row format (attributes + `geometry_wkt` + `geometry_type`) but features form a connected network.

Loaded into a separate `geo_data` dict so it doesn't conflict with the random-sample `data` dict.

In [11]:
geo_index = pd.read_csv(SAMPLE_GEO / "_index.csv")
geo_data = {row["label"]: pd.read_csv(SAMPLE_GEO / row["file"], low_memory=False)
            for _, row in geo_index.iterrows()}
print(f"Loaded {len(geo_data)} layers, {sum(len(d) for d in geo_data.values()):,} features")
geo_index[["label", "asset_class", "geometry_type", "n_features_in_geo_sample"]]

Loaded 23 layers, 18,824 features


,label,asset_class,geometry_type,n_features_in_geo_sample
0,GSI Facility Footprints,Green Infrastructure,Polygon,437
1,Water Quality Polygons,Treatment,Polygon,200
2,Misc Structures,Misc,Polygon,79
3,Other GSI Footprints,Green Infrastructure,Polygon,38
4,Vaults,Detention,Polygon,36
5,Pump Stations,Pump Stations,Polygon,6
6,CSO Basins,Service Areas,MultiPolygon,2
7,SPU-Owned Non-Mainlines,Conveyance,LineString,5252
8,Mainlines (Permitted Use),Conveyance,LineString,2935
9,GSI Underdrains & Rain Gardens,Green Infrastructure,LineString,286


## 6. Multi-layer interactive map (geographic sample)

Stacks polygons, lines, and points from `geo_data` on one folium map. The `LayerControl` widget in the corner lets you toggle each layer on/off. Polygon features (basins, footprints) sit on the bottom; line features (mainlines, ditches) in the middle; point features (catch basins, outfalls) on top.

In [ ]:
if HAS_GIS:
    import folium

    def add_layer(m, gdf, name, color, opacity=0.6, point_radius=3):
        if gdf.geometry.iloc[0].geom_type == "Point":
            for _, row in gdf.iterrows():
                folium.CircleMarker(
                    location=[row.geometry.y, row.geometry.x],
                    radius=point_radius, color=color, fill=True, fill_opacity=0.8,
                    popup=name,
                ).add_to(m)
        else:
            folium.GeoJson(
                gdf.to_json(),
                name=name,
                style_function=lambda _: {"color": color, "weight": 2, "fillOpacity": opacity * 0.3},
                tooltip=name,
            ).add_to(m)

    # Pull the layers we care about — order matters: polygons first (bottom), points last (top)
    layer_specs = [
        ("CSO Basins",                  "#7F77DD"),  # purple polygons
        ("GSI Facility Footprints",     "#5DCAA5"),  # teal polygons
        ("Vaults",                      "#1D9E75"),  # dark teal
        ("Pump Stations",               "#534AB7"),
        ("Mainlines (Permitted Use)",   "#185FA5"),  # blue lines
        ("SPU-Owned Non-Mainlines",     "#85B7EB"),  # light blue lines
        ("Ditches & Culverts",          "#0F6E56"),
        ("GSI Swales",                  "#3B6D11"),
        ("Storm Outfalls",              "#993C1D"),  # coral points
        ("Catch Basins",                "#D85A30"),
    ]

    # Center the map on the bbox center
    center = [(47.575 + 47.610) / 2, (-122.350 + -122.315) / 2]
    m2 = folium.Map(location=center, zoom_start=14, tiles="cartodbpositron")

    for label, color in layer_specs:
        if label not in geo_data:
            continue
        gdf = to_gdf(geo_data[label])
        if len(gdf) == 0:
            continue
        add_layer(m2, gdf, label, color)

    folium.LayerControl(collapsed=False).add_to(m)

# ---- Legend ----
    legend_html = ['''
    <div style="
        position: fixed;
        bottom: 30px; left: 30px;
        z-index: 9999;
        background: white;
        padding: 12px 14px;
        border: 1px solid #888;
        border-radius: 6px;
        font-family: -apple-system, sans-serif;
        font-size: 12px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.15);
        max-width: 260px;
    ">
      <div style="font-weight: 600; margin-bottom: 8px;">Asset layers</div>
    ''']
    for label, color in layer_specs:
        if label not in geo_data or len(geo_data[label]) == 0:
            continue
        count = len(geo_data[label])
        legend_html.append(
            f'<div style="display: flex; align-items: center; gap: 8px; margin: 3px 0;">'
            f'<span style="width: 12px; height: 12px; background: {color}; '
            f'border: 0.5px solid #333; flex-shrink: 0;"></span>'
            f'<span>{label} <span style="color: #888;">({count:,})</span></span>'
            f'</div>'
        )
    legend_html.append('</div>')
    m2.get_root().html.add_child(folium.Element("".join(legend_html)))
    
    m2
else:
    print("Run the mapping-setup cell first (shapely / geopandas / folium needed).")

In [17]:
m2.save("map_v2.html")
webbrowser.open("file://" + os.path.abspath("map_v2.html"))

True

## 7. Where to go from here

- **Cross-sectional probes**: lifecycle status by asset class, install-date distributions, ownership splits between SPU vs private, NPDES coverage rates, dimensions vs material — use the `data` dict (random sample).
- **Spatial probes**: outfalls per drainage basin, GSI density vs catch basin density, distance from each outfall to nearest mainline endpoint — use the `geo_data` dict (geographic sample).
- **Network probes**: join mainlines to their endpoints (via `MNL_UPS_ENDPT_ID` / `MNL_DNS_ENDPT_ID`) to reconstruct topology; trace upstream contributors to a given outfall.
- **Different neighborhood**: edit `BBOX` at the top of `build_geo_sample.py` and rerun to produce a `sample_geo/` for any other area of Seattle.
- **Promotion to the full dataset**: once a probe is shaped well on the sample, swap `SAMPLE` to `~/Desktop/spu/geojson/` (the full `.geojson.gz` files) and use `geopandas.read_file()` instead of `pd.read_csv`.